# OMX Helsinki - yhtenäinen analyysidatasetti

Tämä notebook käyttää `src/stock_analysis`-pakettia, hakee jokaiselle tickerille yhden rivin ja tallentaa lopullisen datasetin vain kerran. Yksittäisen osakkeen hakuvika ei pysäytä koko ajoa eikä poista riviä datasetistä.

## 1. Asetukset

Muuta vuosia tässä solussa ennen ajoa.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from stock_analysis.config import AnalysisConfig
from stock_analysis.pipeline import run_analysis, save_dataset

CONFIG = AnalysisConfig(
    ticker_file=PROJECT_ROOT / 'Ticker_symbols.xlsx',
    max_workers=4,
    output_file=PROJECT_ROOT / 'stock_analysis_dataset.xlsx',
)
CONFIG

## 2. Datan muodostaminen

Tämä on ajon ainoa verkkohakuosuus. Se voi kestää useita minuutteja.

In [ ]:
dataset = run_analysis(CONFIG)
print(f'Rivejä: {len(dataset)}')
print(dataset['status'].value_counts(dropna=False))
dataset.head()

## 3. Laadun tarkistus ja suodatus

Kaikki lähdetiedoston tickerit säilyvät datasetissä. Tarkista ensin virherivit ja suodata sen jälkeen haluamasi osakkeet.

In [ ]:
error_rows = dataset[dataset['status'].ne('ok')]
print(f'Virherivejä: {len(error_rows)}')
error_rows[['symbol', 'yahoo_ticker', 'error']].head(20)

In [ ]:
# Esimerkki: yhtiöt, joiden ROE on positiivinen
filtered = dataset.query("status == 'ok' and roe > 0").sort_values('roe', ascending=False)
filtered[['symbol', 'name', 'roe', 'pe_ratio', 'pb_ratio']].head(20)

## 4. Lopullinen vienti

Vienti tehdään vain kerran. `Dataset` sisältää analyysirivit ja `Run_metadata` ajon parametrit.

In [ ]:
output_path = save_dataset(dataset, CONFIG)
print(f'Tallennettu: {output_path}')

In [ ]:
dataset